### Estructuras 
- Nodo
- Nodo con costes

In [ ]:
class Node:
    def __init__(self, state, parent=None):
        self.state = state
        self.parent = parent
    def __str__(self):
        return f"Node(state={self.state}, parent={self.parent})"    

class NodeCost:
    def __init__(self, state, cost, parent=None):
        self.state = state
        self.cost = cost
        self.parent = parent

    def __lt__(self, other):
        return self.cost < other.cost

    def __str__(self):
        return f"Node(state={self.state}, parent={self.parent}, cost ={self.cost})"        

### Algoritmos

- breadth_first_search
- uniform_cost_search
- depth_first_search

In [ ]:
from collections import deque


def imprimirFrontera (frontera):
    print("\t\tFRONTERA: ")    
    for node in frontera:
        print(f"\t\t\t{node}")
        
def breadth_first_search(problem):
    longitud_frontera = 0 
    node = Node(problem.initial_state)
    if problem.is_goal(node.state):
        return node
    
    frontier = deque([node])                                    # Cola para BFS
    reached = {tuple(problem.initial_state)}
    
    while frontier:
        node = frontier.popleft()                               # Extrae el nodo más antiguo (FIFO)
        print (f"NODO SELECCIONADO {node}")
        for child_state in problem.expand(node.state):
            print (f"\tHIJO EXPANDIDO {child_state}")
            if tuple(child_state) not in reached:
                child_node = Node(child_state, node)
                if problem.is_goal(child_state):
                    print (f"SOL={child_state}")
                    print (f"max frontier longitud = {longitud_frontera}")
                    return child_node
                reached.add(tuple(child_state))
                frontier.append(child_node)                    # Agrega el nodo hijo al final de la cola
                imprimirFrontera(frontier)
                if (longitud_frontera < len(frontier)):
                    longitud_frontera = len(frontier)

    return "failure"

import heapq

def uniform_cost_search(problem):
    initial_node = NodeCost(problem.initial_state, 0)
    if problem.is_goal(initial_node.state):
        return initial_node

    frontier = []
    heapq.heappush(frontier, initial_node)              # Cola de prioridad para UCS
    reached = {problem.initial_state: 0}

    while frontier:
        node = heapq.heappop(frontier)                   # Extrae el nodo con el costo más bajo
        if problem.is_goal(node.state):
            return node

        for child_state, step_cost in problem.expand(node.state):
            child_cost = node.cost + step_cost
            if child_state not in reached or child_cost < reached[child_state]:
                child_node = NodeCost(child_state, child_cost, node)
                reached[child_state] = child_cost
                heapq.heappush(frontier, child_node)    # Agrega el nodo hijo a la frontera con su costo actualizado

    return "failure"

def depth_first_search(problem):
    reached = set()
    frontier = [Node(problem.initial_state)]            #   Pila para DFS

    while frontier:
        node = frontier.pop()                           # Extrae el nodo más reciente (LIFO)
        print (f"NODO SELECCIONADO {node}")
        if problem.is_goal(node.state):
            return node
        if tuple(node.state) not in reached:
            reached.add(tuple(node.state))
            children = [Node(child_state, node) for child_state in problem.expand(node.state)]
            frontier.extend(children)                   # Agrega los nodos hijos a la pila
            imprimirFrontera(frontier)

    return None

## Problema Puzzle 8

- Observa el funcionamiento del algoritmo en amplitud.
- Ejecuta el algoritmo y observa como se van explorando los distintos nodos.
- Para la creación de la solución necesitamos crear un camino desde el nodo objetivo hasta el nodo inicial.
- El problema del puzzle 8 está definido fuera del algoritmo de búsqueda en una clase Puzzle8Problem.
- Crea una clase Puzzle15Problem para realizar el problema del puzzle 15 y ejecuta el algoritmo.
- Observa longitud_frontera de cada problema.

In [ ]:
from collections import deque

class Puzzle8Problem:
    def __init__(self, initial_state, goal_state):
        self.initial_state = initial_state
        self.goal_state = goal_state

    def is_goal(self, state):
        return state == self.goal_state

    def expand(self, state):
        children = []
        zero_index = (state.index(0) // 3, state.index(0) % 3)
        moves = [(1, 0), (-1, 0), (0, 1), (0, -1)]              # Movimientos: arriba, abajo, izquierda, derecha
        for move in moves:
            new_index = (zero_index[0] + move[0], zero_index[1] + move[1])
            if 0 <= new_index[0] < 3 and 0 <= new_index[1] < 3:  # Verificar límites
                new_state = list(state)
                new_state[zero_index[0] * 3 + zero_index[1]], new_state[new_index[0] * 3 + new_index[1]] = \
                    new_state[new_index[0] * 3 + new_index[1]], new_state[zero_index[0] * 3 + zero_index[1]]
                children.append(new_state)
        return children


# Ejemplo de uso
#initial_state = [1, 2, 3, 0, 4, 6, 7, 5, 8]
initial_state = [1, 2 , 0, 3, 4, 5, 6, 7, 8]
goal_state = [0, 1, 2, 3, 4, 5, 6, 7, 8]
problem = Puzzle8Problem(initial_state, goal_state)
solution_node = breadth_first_search(problem)

def imprimirTablero (estado):
    for i in range(3):
        print(estado[i * 3:i * 3 + 3])
    print()

def print_solution (solution_node):
    # Reconstruir la ruta hacia la solución
    if solution_node != "failure":
        path = []
        while solution_node:
            path.append(solution_node.state)
            solution_node = solution_node.parent
        path.reverse()
        print("\n\nSolución encontrada:\n")
        for state in path:
            #imprimirTablero(state)
            print(state)
    else:
        print("No se encontró solución.")

print_solution (solution_node)       


### Problema de Grafo

In [ ]:
class ProblemGrafo:
    def __init__(self, initial_state, goal_state):
        self.initial_state = initial_state
        self.goal_state = goal_state

    def is_goal(self, state):
        return state == self.goal_state

    def expand(self, state):
        # Define cómo expandir un estado para obtener los posibles estados hijos
        sucesores = {
            '1': ['2', '3'],
            '2': ['4', '6'],
            '3': ['4', '5', '7'],
            '4': ['5'],
            '5': ['6', '7'],
            '6': ['7', '8'],
            '7': ['8']
        }
        return sucesores.get(state, [])  # Devuelve los vecinos del estado o una lista vacía si el estado no tiene vecinos

  
initial_state = '1'
goal_state = '8'
problem = ProblemGrafo(initial_state, goal_state)
solution_node = breadth_first_search(problem)

if solution_node != "failure":
    print("Solución encontrada:")
    current_node = solution_node
    print(current_node.state)
    
    while current_node.parent is not None:
        print(current_node.parent.state)
        current_node = current_node.parent
else:
    print("No se encontró solución.")

# Dijkstra o Coste Uniforme

- Hace uso de una cola de prioridad (heapq)
- De la misma forma que el anterior hemos definido una clase  ProblemRumaniaMapa que reproduce el problema del libro.
- Define un problema ProblemAndalucia que realice obtenga un punto inicial y un objetivo y me muestre el coste y el camino de ir del punto inicial al objetivo.

In [ ]:

class ProblemRumaniaMapa:
    def __init__(self, initial_state, goal_state):
        self.initial_state = initial_state
        self.goal_state = goal_state

    def is_goal(self, state):
        return state == self.goal_state

    def expand(self, state):
        # Define cómo expandir un estado para obtener los posibles estados hijos y sus costos
        sucesores = {
            'Sibiu': {'Rimnicu Vilcea': 80, 'Fagaras': 99},
            'Rimnicu Vilcea': {'Sibiu': 80, 'Pitesti': 97},
            'Fagaras': {'Sibiu': 99, 'Bucharest': 211},
            'Pitesti': {'Rimnicu Vilcea': 97, 'Bucharest': 101},
            'Bucharest': {'Fagaras': 211, 'Pitesti': 101}
        }
        if state in sucesores:
            return [(child_state, step_cost) for child_state, step_cost in sucesores[state].items()]
        else:
            return []  # Si el estado no tiene sucesores, retornar una lista vacía

# Ejemplo de uso
initial_state = 'Sibiu'
goal_state = 'Bucharest'
problem = ProblemRumaniaMapa(initial_state, goal_state)
solution_node = uniform_cost_search(problem)

if solution_node != "failure":
    print(f"Coste solución = {solution_node.cost}")    
    print("Solución encontrada:")
    current_node = solution_node
    print(current_node.state)
    
    while current_node.parent is not None:
        print(current_node.parent.state)
        current_node = current_node.parent
else:
    print("No se encontró solución.")


### Grafo en Profundidad

- Plantea  el problema del grafo visto anteriormente
- Resuelve el problema usando el algoritmo de búsqueda en Profundidad

### Puzzle 8 Profundidad

- Plantea  el problema del puzzle 8 y puzzle 15
- Resuelve el problema usando el algoritmo de búsqueda en Profundidad
